### **Extraer el `titulo`, `reseña`, `transcripción` y `enlace de la pelicula` de cada pelicula que comience con X que se encuentre en la primera página y exportarlos a un archivo `JSON`**

Vamos a trabajar en el enlace:

https://subslikescript.com/movies_letter-X

#### **Crear la `crawl spider`**

Ahora vamos a crear un nuevo spider utilizando una crawl template. Así que escribimos "**`scrapy genspider -t crawl`**" y luego escribimos el nombre de la spider que queremos crear. 

<center><img src="https://i.postimg.cc/tT6qKRKb/ws-449.png"></center>

Luego, debemos editar el archivo `transcripcion.py` que se crea, con el código que se muestra abajo.

#### **Paso a paso**

Vamos a inspeccionar el sitio:

<center><img src="https://i.postimg.cc/Dz5cbH23/ws-142.png"></center>

Vamos a localizar los enlaces a cada una de las peliculas de la pagina:

<center><img src="https://i.postimg.cc/3wdJZF9d/ws-448.png"></center>

Y vamos a escoger cualquier pelicula para inspeccionar su página:

<center><img src="https://i.postimg.cc/zBBFw7Bb/ws-143.png"></center>

Localizamos la caja que encierre todo el contenido de la página:

<center><img src="https://i.postimg.cc/BbP5kv2J/ws-144.png"></center>

Localizamos su `titulo`:

<center><img src="https://i.postimg.cc/mZP0BNwg/ws-145.png"></center>

Localizamos su `introducción`:

<center><img src="https://i.postimg.cc/nM8KJt1B/ws-146.png"></center>

Y finalmente, localizamos su `transcripción`:

<center><img src="https://i.postimg.cc/3RcCtG4L/ws-147.png"></center>

#### **Código**

Pero ahora tenemos una nueva propiedad llamada "**`rules`**", que es una tupla. Una tupla funciona como una lista, pero son inmutables, lo que significa que una vez que la configuras, no puedes cambiar lo que hay dentro de ella. La tupla "rules" debe contener al menos un objeto regla. El objeto "rules" se utiliza para indicar a la crawl spider uno de los enlaces que quiere seguir (follow) en el sitio web que estamos scrapeando. Por defecto, ya tenemos una regla configurada. Esta regla tiene tres argumentos. El primer argumento es "**`allow`**". El segundo es el método "**`callback`**", que **`se establece como un string`**. A diferencia de lo que hemos visto en la plantilla básica, el método callback en la crawl spider debe definirse como un string. El último argumento es el argumento "**`follow`**". Es un argumento booleano y se establece a "**`true`**" y enviará una solicitud (request) a los enlaces (links) extraídos. Finalmente, tenemos el objeto "**`LinkExtractor`**". Sirve para especificar los enlaces que queremos extraer o no. Para definir los enlaces que queremos extraer, utilizamos el argumento "allow". En este caso, se indica una expresión regular: `r'Items/'`. Esto significa que si el enlace contiene la palabra "items" seguida y luego parsear la respuesta en la función "`parse_item()`". Lo contrario de este argumento es el argumento "**`deny`**". En este caso, si el enlace contiene la palabra "item", entonces será seguido. También existe el argumento "**`restrict_xpaths`**". Este restringe la región que queremos que la crawl spider extraiga los enlaces. 

In [ ]:
import scrapy
from scrapy.linkextractors import LinkExtractor
from scrapy.spiders import CrawlSpider, Rule


class TranscripcionSpider(CrawlSpider):
    name = "transcripcion"
    allowed_domains = ["subslikescript.com"]
    start_urls = ["https://subslikescript.com/movies_letter-X"]

    # Establecer reglas para el crawler
    # Así que seguiremos sólo los enlaces de los elementos que tengan este XPath (restrict_xpaths).
    # Además, como se puede ver, no he añadido el "/@href" (//ul[@class='scripts-list']/a/@href) esto es porque el 
    # objeto "LinkExtractor" buscará automáticamente el atributo "href", por lo que tenemos que omitirlo.
    rules = (
        Rule(LinkExtractor(restrict_xpaths=("//ul[@class='scripts-list']/a")), callback='parse_item', follow=True),
    )

    def parse_item(self, response):
        # Obtener la caja del article que contiene los datos que queremos (title, plot, etc)
        article = response.xpath("//article[@class='main-article']")

        # Extraer los datos que queremos y luego devolverlos
        yield {
            'titulo': article.xpath("./h1/text()").get(),
            'reseña': article.xpath("./p/text()").get(),
            'transcripcion': article.xpath("./div[@class='full-script']/text()").getall(),
            'enlace': response.url,
        }

#### **Ejecución**

Nos posicionamos en la ruta correcta y ejecutamos nuestra Spider:

In [ ]:
scrapy crawl transcripcion

<center><img src="https://i.postimg.cc/zf6D2HPp/ws-450.png"></center>

De esta manera obtenemos toda la info que pedimos extraer del website:

<center><img src="https://i.postimg.cc/D0KvnkhX/ws-451.png"></center>

Podriamos exportar toda esta data en un archivo `JSON` escribiendo el siguiente comando:

In [ ]:
scrapy crawl transcripcion -o transcripcion.json